In [7]:
# Output a file for manually uploading to an AI to ask for a returned JSON file with a mapping of different names how they appear in the data to a standardized name.

import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import json
from string import Template

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import utils


In [8]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [9]:
names = {}

In [10]:
# minutes items dataframe

minutes_items_df = []
for _, row in meetings_df.iterrows():
    date = row['date']
    year = date[0:4]
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized-minutes.json")
    with open(input_filepath, 'r', encoding='utf-8') as f:
        minutes_items = json.load(f)
    for item in minutes_items:
        if not item['ai_minutes_summary']:
            continue
        mover = item['ai_minutes_summary'].get('mover', "")
        if mover:
            names[mover] = names.get(mover, 0) + 1
        seconder = item['ai_minutes_summary'].get('seconder', "")
        if seconder:
            names[seconder] = names.get(seconder, 0) + 1
        ayes = item['ai_minutes_summary'].get('ayes', [])
        for name in ayes:
            names[name] = names.get(name, 0) + 1
        noes = item['ai_minutes_summary'].get('noes', [])
        for name in noes:
            names[name] = names.get(name, 0) + 1
        abstentions = item['ai_minutes_summary'].get('abstentions', [])
        for name in abstentions:
            names[name] = names.get(name, 0) + 1
        absences = item['ai_minutes_summary'].get('absences', [])
        for name in absences:
            names[name] = names.get(name, 0) + 1
        recusals = item['ai_minutes_summary'].get('recusals', [])
        for name in recusals:
            names[name] = names.get(name, 0) + 1


In [11]:
names_df = pd.DataFrame(list(names.items()), columns=['name', 'count'])
names_df = names_df.sort_values(by=['name', 'count'], ascending=[True, False]).reset_index(drop=True)

output_filepath = os.path.join(LOCAL_PATH, "intermediate_data/cpc/names-to-standardize.json")
with open(output_filepath, 'w', encoding='utf-8') as f:
    json.dump(names, f, indent=2, ensure_ascii=False)

In [12]:
# Take the output file and upload it to an AI. Prompt it with this message:
"""
Here is a JSON file showing different last names of Los Angeles City Planning Commission members, as they appear in a dataset, along with the number of times each appears in the data. Return a mapping of the different names to a standardized name. For example, "Dake-Wilson" should be standardized to "Dake Wilson". Most of the variations are due to hyphenation and accents. When the same person has multiple variations of their name, choose the most common variation as the standardized name. Return the mapping as a JSON object, where the keys are the different names as they appear in the data, and the values are the standardized names.
"""
# Then save the resulting JSON to "LOCAL_PATH/intermediate_data/cpc/names-standardized.json" and use it on downstream tasks


'\nHere is a JSON file showing different last names of Los Angeles City Planning Commission members, as they appear in a dataset, along with the number of times each appears in the data. Return a mapping of the different names to a standardized name. For example, "Dake-Wilson" should be standardized to "Dake Wilson". Most of the variations are due to hyphenation and accents. When the same person has multiple variations of their name, choose the most common variation as the standardized name. Return the mapping as a JSON object, where the keys are the different names as they appear in the data, and the values are the standardized names.\n'